In [120]:
# Import and create models
import TailKinematicsNN
import torch

TailKinematicsModel = TailKinematicsNN.TailKinematicsRNN()
TailKinematicsModel.load_state_dict(torch.load('./Models/kinematicsMoments_0.9991245342.pt'))

# count parameters
total_params = sum(p.numel() for p in TailKinematicsModel.parameters() if p.requires_grad)
print(f'Total trainable parameters in TailKinematicsModel: {total_params}')

Total trainable parameters in TailKinematicsModel: 172


In [121]:
# load Matlab data
import scipy.io
import numpy as np

data = scipy.io.loadmat('./Data/data2025-10-18_00-01-54.mat')
data_keys = ['xout', 'Tail_Forces_out', 'Tail_Centre_out', 'cout', 'cal_stout', 'Moments_Add_out']

# Create a dictionary to hold the data (labels as keys and numpy arrays as values)
data_dict = {key: np.array(data[key]) for key in data_keys}

In [122]:
print(data_dict['cal_stout'].shape)  # Example: print the shape of 'cal_stout' data
data_dict['cal_stout'] = data_dict['cal_stout'][:, :8]  # Keep only the first 8 columns

# Prepare input kinematics tensor (8 motor angular positions + tail centre position))
in_kinematics = np.hstack((data_dict['cal_stout'], data_dict['Tail_Centre_out']))

kinematics_tensor = torch.from_numpy(in_kinematics).float().unsqueeze(-1)  # Add batch dimension
print(kinematics_tensor.shape)

out_kinematics = np.hstack((data_dict['Moments_Add_out'], data_dict['cout'])) # Target output: moments + caudal amplitude + angle
kinematics_target = torch.from_numpy(out_kinematics).float()
print(kinematics_target.shape)

(10000, 10)
torch.Size([10000, 9, 1])
torch.Size([10000, 3])


In [123]:
import time
# Get prediction time
start_time = time.time()
TailKinematicsModel.eval()
with torch.no_grad():
    kinematics_preds = TailKinematicsModel(kinematics_tensor)
    print(kinematics_preds.shape)
end_time = time.time()
total_time = end_time - start_time
mean_time_per_prediction = total_time / kinematics_preds.shape[0]
print(f"Prediction total time: {total_time} seconds")
print(f"Mean prediction time: {mean_time_per_prediction} seconds")

torch.Size([10000, 3])
Prediction total time: 0.03158879280090332 seconds
Mean prediction time: 3.158879280090332e-06 seconds


In [124]:
# Test compilted model
compiled_model = torch.compile(TailKinematicsModel)
start_time = time.time()
compiled_model.eval()
with torch.no_grad():
    kinematics_preds_compiled = compiled_model(kinematics_tensor)
    print(kinematics_preds_compiled.shape)
end_time = time.time()
total_time = end_time - start_time
mean_time_per_prediction = total_time / kinematics_preds_compiled.shape[0]
print(f"Compiled model prediction total time: {total_time} seconds")
print(f"Compiled model mean prediction time: {mean_time_per_prediction} seconds")

torch.Size([10000, 3])
Compiled model prediction total time: 0.027273178100585938 seconds
Compiled model mean prediction time: 2.7273178100585937e-06 seconds


In [125]:
# compare single sample prediction times
import time
TailKinematicsModel.eval()
start_time = time.time()
for data in kinematics_tensor:
    with torch.no_grad():
        single_pred = TailKinematicsModel(data.view(1, -1, 1))
end_time = time.time()
print(f"Single sample prediction time: {(end_time - start_time)/kinematics_tensor.shape[0]} seconds")

# Compiled model
start_time = time.time()
for data in kinematics_tensor:
    with torch.no_grad():
        single_pred_compiled = compiled_model(data.unsqueeze(0))
end_time = time.time()
print(f"Single sample prediction time: {(end_time - start_time)/kinematics_tensor.shape[0]} seconds")


Single sample prediction time: 0.000766759991645813 seconds
Single sample prediction time: 0.0010813109874725341 seconds


In [ ]:
import ThrustNN

input_size = 16
hidden_layers =5
npl = 64
output_size = 7
ThrustModel = ThrustNN.thrustFlexNN(input_size=input_size, hidden_size=npl, output_size=output_size, hidden_layers=hidden_layers, dropout_enabled=True)
ThrustModel.load_state_dict(torch.load('./Models/simple_data_modelv2_64_neurons_5_layers.pt'))

# Compile model
compiled_thrust_model = torch.compile(ThrustModel)

# Get traced model
traced_thrust_model = torch.jit.trace(ThrustModel, torch.randn(1, input_size))

c:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\jit\_trace.py:1307: TracerWarning: Trace had nondeterministic nodes. Did you forget call .eval() on your model? Nodes:
	%input.5 : Float(1, 64, strides=[64, 1], requires_grad=1, device=cpu) = aten::dropout(%input.3, %39, %40) # c:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\functional.py:1425:0
	%input.11 : Float(1, 64, strides=[64, 1], requires_grad=1, device=cpu) = aten::dropout(%input.9, %44, %45) # c:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\functional.py:1425:0
	%input.17 : Float(1, 64, strides=[64, 1], requires_grad=1, device=cpu) = aten::dropout(%input.15, %49, %50) # c:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\functional.py:1425:0
	%input.23 : Float(1, 64, strides=[64, 1], requires_grad=1, device=cpu) = aten::dropout(%input.21, %54, %55) # c:\Users\brend\AppData\Local\Programs\Python\Python3

: 

In [127]:
# Prepare input for Thrust Model with simulation data
#load scalers used during training
import joblib
scaler_in = joblib.load('./Scalers/scaler_thrust_in.pkl')
scaler_out = joblib.load('./Scalers/scaler_thrust_out.pkl')
# Concatenate inputs for Thrust Model
caudal_old = np.hstack((np.zeros(1),data_dict['cout'][:-1, 0])).reshape(-1,1)  # Previous caudal angle (shifted by one timestep)
print(caudal_old.shape)
in_thrust = np.hstack((data_dict['xout'], data_dict['cout'], caudal_old, data_dict['Tail_Centre_out']))  # Input: fluid velocity, caudal angle, previous caudal angle, tail forces
print(in_thrust.shape)
# Scale input data 
in_thrust = scaler_in.transform(in_thrust)  # Scale input data
# Convert to tensor
in_thrust_tensor = torch.from_numpy(in_thrust).float()

# Prepare target output for Thrust Model
thrust_target = data_dict['Tail_Forces_out']  # Target output: thrust forces
print(thrust_target.shape)
# Exclude all columns that are zero constants
thrust_target = thrust_target[:, ~np.all(thrust_target == 0, axis=0)]
# Scale target data
thrust_target = scaler_out.transform(thrust_target)  # Scale target data
# Convert to tensor
thrust_target_tensor = torch.from_numpy(thrust_target).float()
print(thrust_target_tensor.shape)

(10000, 1)
(10000, 16)
(10000, 12)
torch.Size([10000, 7])


In [128]:
# Compare normal vs compiled model prediction times
import time
start_time = time.time()
with torch.no_grad():
    TailKinematicsModel.eval()
    kinematics_preds = TailKinematicsModel(kinematics_tensor)
end_time = time.time()
total_time = end_time - start_time
mean_time_per_prediction = total_time / kinematics_preds.shape[0]
print(f"Normal model prediction total time: {total_time} seconds")
print(f"Normal model mean prediction time: {mean_time_per_prediction} seconds")

Normal model prediction total time: 0.03697395324707031 seconds
Normal model mean prediction time: 3.697395324707031e-06 seconds


In [129]:
# Compare normal vs compiled model prediction times
import time
start_time = time.time()
with torch.no_grad():
    compiled_model.eval()
    kinematics_preds = compiled_model(kinematics_tensor)
end_time = time.time()
total_time = end_time - start_time
mean_time_per_prediction = total_time / kinematics_preds.shape[0]
print(f"Compiled model prediction total time: {total_time} seconds")
print(f"Compiled model mean prediction time: {mean_time_per_prediction} seconds")

Compiled model prediction total time: 0.036934614181518555 seconds
Compiled model mean prediction time: 3.6934614181518555e-06 seconds
